# Langflow Agent with memory, unboxed

This notebook runs the card **Langflow Agent with memory chat · local Qwen3.8-27B** one step per cell, with the card's own system prompt, its model on this machine, and its tools. Each cell prints what was sent and what came back; a folded line under it opens the code that ran.

| On the Langflow canvas | In this notebook |
|---|---|
| The Agent card calls the model and runs tools by itself | `agent.decide()` is one model call: its thinking, then the tool it chose. `agent.act()` runs that one tool. |
| A Run Flow card runs a whole flow | The flow's cards run one after another, each with its receipt |
| The Design pipeline is one tool call | One cell per card: start, memory, requirements, router, the writer, the Pilot, alignment, checks, router, trade-offs, review |
| Inside the pipeline, OpenCode writes design A and B in a sandbox | The **Langflow Agent, Pilot only** card writes them instead, driven here one turn per cell: `w.decide()` writes, `w.act()` is one official-Pilot call, the verdict goes back to the model |
| Write to memory shows Approve and Reject | `agent.approve(True)` or `agent.approve(False)` |

The writer is the one place this notebook departs from the canvas. The canvas's Design pipeline card hands A and B to OpenCode, an autonomous process whose own `./validate` calls cannot be paused between. Here the same work is given to the Langflow Agent, Pilot only card (project 2) with the same task text the pipeline hands every writer, so every Pilot round is a cell. On the Part 2 benchmark the two writers score the same tier (OpenCode 249/300, the Langflow agents 246/298 and 251/300). Everything before and after the writer is the same shell step the canvas calls, and the pipeline still re-parses what the writer hands in (④), as it does for OpenCode.

Two smaller differences: the knowledge-base card is left out (it searches Langflow's own vector store; Memory search reads the same cards), and the canvas does not light up. Google Colab cannot run this: the model, the shell service, the MCP server and the Pilot are on this machine.

## 0 · The machine

Start the services once, in a terminal: `python3 scripts/up.py --bg`. That starts the shell service, the MCP server, Ollama and Langflow. The results page may stay open.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bench" / "shell_service.py").is_file())
sys.path.insert(0, str(ROOT))
from notebooks.agent_walk import Agent, check_services

check_services()

  ok  RTRC shell service                 http://127.0.0.1:8781 · results page open: True
  ok  RTRC MCP server                    http://127.0.0.1:8782/mcp · 9 tools
  ok  Ollama and the card's model        qwen3.8-27b-q3:64k installed · in GPU memory now
  ok  Official Pilot 0.62 (jar)          mvp/toolchain/kernel/sysml/jupyter-sysml-kernel-0.62.0-all.jar
  ok  OpenCode (pinned binary)           tools/opencode/opencode
  ok  Sandbox: bubblewrap                /usr/bin/bwrap
  ok  Sandbox network: slirp4netns       /usr/bin/slirp4netns

Everything the agent needs is up.


True

## 1 · The Agent card, unboxed

What the model receives with every call: the card's system prompt and its tools.

In [2]:
agent = Agent()
agent.show_card()


━━ The Agent card · Langflow Agent with memory chat · local Qwen3.8-27B · Langflow Agent · local Qwen3.8-27B ━━━━
Flow file        langflow/flows/4-langflow-agent-with-memory/langflow-agent-memory-chat-local.json
Model            qwen3.8-27b-q3:64k · provider Ollama · http://127.0.0.1:11434
Messages kept    30 · model calls per turn at most 20 · a failing tool's error goes back to the model: True
System prompt    3,871 characters (open it below)

Tools the model is given (9), as the canvas wires them to the Agent's Tools port:
   1. Memory-search_tool                             Run Flow · flow "Memory search" · Memory library → MiniLM + FAISS seeds → One hop of neighbours -> cards for the assistant
   2. Design-pipeline_tool                           Run Flow · flow "Design pipeline" · start → ① memory → ② requirements → new | adapt → ③ OpenCode designs → ④ Pilot → ⑤ align → ⑥ checks → trade-offs? → ⑦ review
   3. Validate-SysML-with-the-official-Pilot_tool    Run Flow · flow "Valida

## 2 · Turn 1 · the engineer asks

The engineer gives a need with all its numbers and asks the agent to look in memory first. The numbers are the brief confirmed in the Playground on 13 October.

Two separate cells follow: the model chooses, then the chosen tool runs.

In [3]:
agent.say("""I want a new design for this fixed-wing UAV. First check the memory for anything similar, then show me the exact brief you would send to the design pipeline.

Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.""")


━━ Engineer ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
I want a new design for this fixed-wing UAV. First check the memory for anything similar, then show me the exact brief you would send to the design pipeline.

Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.


In [4]:
agent.decide()


━━ Step 1 · the model reads 2 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 user

 is

 asking

...

 the

 calls

.


← 12.6 s · read 2496 tokens · wrote 999 tokens · stop: stop

The model chose (nothing has run yet; agent.act() runs it):
  Memory-search_tool({"flow_tweak_data": {"ChatInput-query~input_value": "fixed-wing UAV lidar EO camera search radar edge computer data radio power storage upload"}})
  list_projects({})


{'content': '',
 'thinking': 'The user is asking for two things:\n1. Check the memory for anything similar to this fixed-wing UAV design.\n2. Show the exact brief that would be sent to the design pipeline.\n\nThe user has already provided a fairly complete brief. Let me check what numbers are present:\n- Vessel: fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure\n- Devices:\n  - Mid-360S LiDAR: 6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/sec at 10 Hz\n  - EO camera: 5 W\n  - Search radar: 30 W\n  - Edge computer: 14 W\n  - Data radio: 10 W\n- Power limit: 120 W\n- Local recording rate: 25.6 Mbps\n- Retention: 8 hours\n- Storage: 256 GB\n- Upload rate: 10 Mbps\n- Uplink capacity: 20 Mbps\n\nThis covers all the fields of the brief format:\n- Vessel and devices with power consumption (W) ✓\n- Power limit (W) ✓ 120 W\n- Local recording rate (Mbps) ✓ 25.6 Mbps\n- Retention (hours) ✓ 8 h\n- Storage (GB) ✓ 256 GB\n- Upload rate (Mbps) ✓ 10 Mbps\n- Uplink capacity (Mbp

In [5]:
agent.act()


━━ Step 2 · tool Memory-search_tool · canvas card “Tool · search memory (a flow) · Paper 2” ━━━━━━━━
arguments: {"flow_tweak_data": {"ChatInput-query~input_value": "fixed-wing UAV lidar EO camera search radar edge computer data radio power storage upload"}}
The flow “Memory search” (langflow/flows/5-shared-tools/tool-memory-search.json) runs its 3 cards in order:

[1/3] Memory library: MaritimeISR + approved designs as cards
→ POST http://127.0.0.1:8781/tool/memory/library {}


← ok in 1.6 s
F0 memory: 196 cards in 4 source(s); 538 links
- Existing maritime model MaritimeISR: 63 cards (scenario/maritime-isr/MaritimeISR.sysml)
- Approved design 20261003T144529 · A: 44 cards (mvp/runs/20261015T034622-4db6ff79)
- Approved design 20261003T150116 · A: 45 cards (mvp/runs/20261015T034630-810fcaa8)
- Approved design 20261013T155413 · B: 44 cards (mvp/runs/20261019T070122-a00706c2)

Approved designs in memory: 3



[2/3] MiniLM + FAISS seeds: cosine >= 0.30 + exact-name bonus
→ POST http://127.0.0.1:8781/tool/memory/seeds {"query": "fixed-wing UAV lidar EO camera search radar edge computer data radio power storage upload", "upstream": "F0 memory: 196 cards in 4 source(s); 538 links\n- Existing maritime model MaritimeISR: 63 cards (scenario/maritime-isr/MaritimeISR.sysml)\n…"}


← ok in 1.6 s
Seeds for "fixed-wing UAV lidar EO camera search radar edge computer data radio power stora": 4 of 196 cards · sentence-transformers/all-MiniLM-L6-v2 · cosine ≥ 0.3 plus the exact-name bonus 0.1
- 0.50 · PartUsage MaritimeISR::UnmannedSurfaceVessel::radio  [MaritimeISR]
- 0.47 · PartDefinition MaritimeISR::Radar  [MaritimeISR]
- 0.47 · PortUsage DesignB::top::lidar::lidarTxCam  [approved 20261013T155413·B]
- 0.46 · PortUsage DesignB::top::lidar::lidarRxDig  [approved 20261013T155413·B]



[3/3] One hop of neighbours -> cards for the assistant
→ POST http://127.0.0.1:8781/tool/memory/hop {"query": "fixed-wing UAV lidar EO camera search radar edge computer data radio power storage upload", "upstream": "Seeds for \"fixed-wing UAV lidar EO camera search radar edge computer data radio power stora\": 4 of 196 cards · sentence-transformers/all-…"}


← ok in 1.6 s
One hop from the seeds: 3 neighbour(s); 7 cards in all · No related values in existing models: new-design path.
- via radio → PartDefinition MaritimeISR::UnmannedSurfaceVessel  [MaritimeISR]
- via Radar → PortUsage MaritimeISR::Radar::tracks  [MaritimeISR]
- via lidarTxCam → PartUsage DesignB::top::lidar  [approved 20261013T155413·B]

Cards returned to the assistant:
- PartUsage MaritimeISR::UnmannedSurfaceVessel::radio
- PartDefinition MaritimeISR::Radar
- [approved 20261013T155413·B] PortUsage DesignB::top::lidar::lidarTxCam
- [approved 20261013T155413·B] PortUsage DesignB::top::lidar::lidarRxDig
- PartDefinition MaritimeISR::UnmannedSurfaceVessel
- PortUsage MaritimeISR::Radar::tracks
- [approved 20261013T155413·B] PartUsage DesignB::top::lidar



Returned to the model as Memory-search_tool's result: 757 characters.


'One hop from the seeds: 3 neighbour(s); 7 cards in all · No related values in existing models: new-design path.\n- via radio → PartDefinition MaritimeISR::UnmannedSurfaceVessel  [MaritimeISR]\n- via Radar → PortUsage MaritimeISR::Radar::tracks  [MaritimeISR]\n- via lidarTxCam → PartUsage DesignB::top::lidar  [approved 20261013T155413·B]\n\nCards returned to the assistant:\n- PartUsage MaritimeISR::UnmannedSurfaceVessel::radio\n- PartDefinition MaritimeISR::Radar\n- [approved 20261013T155413·B] PortUsage DesignB::top::lidar::lidarTxCam\n- [approved 20261013T155413·B] PortUsage DesignB::top::lidar::lidarRxDig\n- PartDefinition MaritimeISR::UnmannedSurfaceVessel\n- PortUsage MaritimeISR::Radar::tracks\n- [approved 20261013T155413·B] PartUsage DesignB::top::lidar'

The rest of the turn: every further choice and tool, until the agent answers. It stops early if the agent calls the design pipeline or a tool that needs approval.

In [6]:
agent.run_until_reply()


━━ Step 3 · tool list_projects · canvas card “Tool · list projects” ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
arguments: {}
→ MCP tools/call list_projects {} · http://127.0.0.1:8782/mcp


← ok in 1.6 s

{
 "projects": [
  {
   "name": "MaritimeISR",
   "label": "Existing maritime model MaritimeISR",
   "cards": 63,
   "path": "scenario/maritime-isr/MaritimeISR.sysml"
  },
  {
   "name": "approved 20261003T144529·A",
   "label": "Approved design 20261003T144529 · A",
   "cards": 44,
   "path": "mvp/runs/20261015T034622-4db6ff79",
   "brief": "A harbor patrol unmanned surface vessel carries a visible camera drawing 28 W and an edge computer drawing 14 W. Add an infrared camera drawing 12 W. The vessel power limit is 60 W. Imagery is recorded locally at 12 Mbps for 8 hours on 256 GB of storage. Imagery is uploaded to shore at 12 Mbps over one 20 Mbps uplink.",
   "numbers": {
    "DesignA::vessel::uploadMbps": 12.0,
    "DesignA::vessel::visibleCamera::drawW": 28.0,
    "DesignA::vessel::edgeComputer::drawW": 14.0,
    "DesignA::vessel::storageGB": 256.0,
    "DesignA::vessel::powerLimitW": 60.0,
    "DesignA::vessel::infraredCamera::drawW": 12.0,
    "DesignA::vessel::ret


Returned to the model as list_projects's result: 3,479 characters.

━━ Step 4 · the model reads 5 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 memory

...

 fresh

 pipeline

 run

?


← 11.4 s · read 5136 tokens · wrote 1063 tokens · stop: stop

No tool call: the text after “answer ›” is the agent's reply, as the Playground shows it.


## 3 · Turn 2 · the engineer confirms; the agent calls the design pipeline

In [7]:
agent.say("Confirmed. Run the design pipeline with exactly that brief.")


━━ Engineer ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Confirmed. Run the design pipeline with exactly that brief.


In [8]:
agent.decide()


━━ Step 7 · the model reads 9 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 user

 confirmed

 the

...

 as

-is

.


← 2.3 s · read 5722 tokens · wrote 227 tokens · stop: stop

The model chose (nothing has run yet; agent.act() runs it):
  Design-pipeline_tool({"flow_tweak_data": {"ChatInput-brief~input_value": "Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity."}})


{'content': '',
 'thinking': "The user confirmed the brief. I'll run the design pipeline with the brief exactly as-is.\n",
 'tool_calls': [{'id': 'oS9zsR16P9aB9Nzf1z7JCjO2DsYcyIc0',
   'function': {'index': 0,
    'name': 'Design-pipeline_tool',
    'arguments': {'flow_tweak_data': {'ChatInput-brief~input_value': 'Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.'}}}}]}

The tool it chose is a flow of its own. This cell opens it, after running any other tool the model asked for first.

In [9]:
pipe = agent.until_pipeline()


━━ Step 8 · tool Design-pipeline_tool · canvas card “Tool · run the design pipeline (a flow) · Papers 2, 3, 1” ━━━━
arguments: {"flow_tweak_data": {"ChatInput-brief~input_value": "Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity."}}

This tool is a flow of its own: langflow/flows/5-shared-tools/tool-design-pipeline.json. Its cards run in this order; the next cells run them one by one.
   · RTRC Start (model qwen3.8-27b (Ollama))
   · ① Search memory · Paper 2
   · ② Requirements
   · Router: similar project found?
   · ③ New design, or ③ Adapt the si

### 3.1 RTRC Start

In [10]:
pipe.start()


━━ Design pipeline · RTRC Start ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/start {"brief": "Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ether…", "agent": "opencode", "model": "qwen3.8-27b (Ollama)", "replay": ""}
← ok in 0.0 s

run 20261021T072601 · live · agent opencode · qwen3.8-27b (Ollama)

Brief:

Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.


{'ok': True,
 'run': '20261021T072601',
 'summary': 'run 20261021T072601 · live · agent opencode · qwen3.8-27b (Ollama)\n\nBrief:\n\nDesign a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.',
 'data': {'brief': 'Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded

### 3.2 ① Search memory · Paper 2

MiniLM + FAISS over the memory cards, seeds plus one hop. If an approved design is similar, the next card's text starts with “reuse”.

In [11]:
pipe.retrieve()


━━ Design pipeline · ① Search memory · Paper 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/retrieve {"run": "20261021T072601"}


← ok in 1.6 s

reuse · run 20261021T072601

① Memory: 196 cards; 6 retrieved (4 seeds, 2 one-hop neighbours); Reusable values found in existing models: reuse path.

Similar project: MaritimeISR (scenario/maritime-isr/MaritimeISR.sysml)
- seed radio · PartUsage · score 0.76 · MaritimeISR
- seed uav · PartUsage · score 0.71 · MaritimeISR
- seed downlinkMbps · AttributeUsage · score 0.71 · MaritimeISR
- seed radio · PartUsage · score 0.70 · MaritimeISR


{'ok': True,
 'run': '20261021T072601',
 'summary': 'reuse · run 20261021T072601\n\n① Memory: 196 cards; 6 retrieved (4 seeds, 2 one-hop neighbours); Reusable values found in existing models: reuse path.\n\nSimilar project: MaritimeISR (scenario/maritime-isr/MaritimeISR.sysml)\n- seed radio · PartUsage · score 0.76 · MaritimeISR\n- seed uav · PartUsage · score 0.71 · MaritimeISR\n- seed downlinkMbps · AttributeUsage · score 0.71 · MaritimeISR\n- seed radio · PartUsage · score 0.70 · MaritimeISR',
 'path': 'reuse',
 'data': {'path': 'reuse',
  'similar_project': {'name': 'MaritimeISR',
   'path': 'scenario/maritime-isr/MaritimeISR.sysml',
   'package': 'MaritimeISR',
   'brief': ''},
  'retrieval': {'method': 'hybrid',
   'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
   'threshold': 0.3,
   'name_bonus': 0.1,
   'index_key': '2cc0612cec2960ba',
   'sources': [{'name': 'MaritimeISR',
     'label': 'Existing maritime model MaritimeISR',
     'path': 'scenario/maritime-isr/M

### 3.3 ② Requirements

Numbered requirements. A number that is not in the brief blocks approval later.

In [12]:
pipe.requirements()


━━ Design pipeline · ② Requirements ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/requirements {"run": "20261021T072601"}


← ok in 12.2 s

reuse · run 20261021T072601 · similar project MaritimeISR

② 9 requirements, 1 questions; every number is in the brief
- R1: The total power consumption of the UAV payload and avionics shall not exceed the total power limit.
- R2: The Mid-360S lidar shall operate with a power draw of 6.5 W.
- R3: The EO camera shall operate with a power draw of 5 W.
- R4: The search radar shall operate with a power draw of 30 W.
- R5: The edge computer shall operate with a power draw of 14 W.
- R6: The data radio shall operate with a power draw of 10 W.
- R7: The lidar point cloud shall be recorded locally at a data rate of 25.6 Mbps.
- R8: The local storage shall retain data for 8 hours.
- R9: The local storage capacity shall be 256 GB.
- question: The brief states 'Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.' These values are not included in the requirements list because the instructions specify that if a number is not explicitly stated as a requireme

{'ok': True,
 'run': '20261021T072601',
 'summary': "reuse · run 20261021T072601 · similar project MaritimeISR\n\n② 9 requirements, 1 questions; every number is in the brief\n- R1: The total power consumption of the UAV payload and avionics shall not exceed the total power limit.\n- R2: The Mid-360S lidar shall operate with a power draw of 6.5 W.\n- R3: The EO camera shall operate with a power draw of 5 W.\n- R4: The search radar shall operate with a power draw of 30 W.\n- R5: The edge computer shall operate with a power draw of 14 W.\n- R6: The data radio shall operate with a power draw of 10 W.\n- R7: The lidar point cloud shall be recorded locally at a data rate of 25.6 Mbps.\n- R8: The local storage shall retain data for 8 hours.\n- R9: The local storage capacity shall be 256 GB.\n- question: The brief states 'Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.' These values are not included in the requirements list because the instructions specify that if

### 3.4 Router: similar project found?

In [13]:
pipe.route_similar()


━━ Design pipeline · Router “Similar project found?” ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The router reads the requirements card's text and asks: does it start with “reuse”?
It starts with “reuse” → True: ③ Adapt the similar project.


'reuse'

### 3.5 ③ Design A · the Langflow Agent writes it, one Pilot round per cell

The writer gets the Pilot-only card's system prompt, the official Pilot as its one tool, and the task text the pipeline hands every writer (shown folded). Then:

- `wa.decide()` — one model call. The model writes SysML and asks for a Pilot call, or hands in.
- `wa.act()` — that one Pilot call: the Validate flow's two cards (save as a file, official Pilot). PASS or FAIL with the error lines goes back to the model.

Repeat the pair until the model hands in. The first cell below shows one round by hand; the next finishes the remaining rounds. On the local 27B a turn takes about a minute because the model thinks before it writes.

In [14]:
wa = pipe.writer("A")


━━ Design pipeline · ③ design A · writer: Langflow Agent, Pilot only chat · Langflow Agent · qwen3.8-flash ━━━━
→ POST http://127.0.0.1:8781/run/prompt {"run": "20261021T072601", "key": "A", "adapt": true}
← ok in 0.0 s

run 20261021T072601

③ prompt for design A: adapt MaritimeISR · 5677 characters · 5 prompt parts

Card file        langflow/flows/2-coding-agents/langflow-agent-pilot-chat.json (on the canvas this card runs qwen3.8-flash; the pipeline sets the writer's model at run time, here qwen3.8-27b-q3:64k)
Model calls      at most 20 for this design (the card's limit)
Its one tool     Validate-SysML-with-the-official-Pilot_tool → flow “Validate SysML with the official Pilot” (langflow/flows/5-shared-tools/tool-validate-sysml.json): save the text as a file → official Pilot
Branch           adapt MaritimeISR


In [15]:
wa.decide()


━━ Design A · writer turn 1 · the model reads 2 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › I

 need

 to

 design

...

 I

 reply

.


← 119.5 s · read 2103 tokens · wrote 12478 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 3007 characters · sha256 97c5185373b9


{'content': "I'll start by drafting the model and validating it with the official Pilot before I reply.\n\n",
 'thinking': 'I need to design a SysML v2 package called DesignA. It\'s a fixed-wing UAV based on the MaritimeISR FixedWingUav structure.\n\nBrief:\n- Fixed-wing UAV\n- Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s @ 10 Hz)\n- EO camera (5 W)\n- Search radar (30 W)\n- Edge computer (14 W)\n- Data radio (10 W)\n- Total power limit 120 W\n- Lidar point cloud recorded locally at 25.6 Mbps, 8-hour retention, 256 GB storage\n- Processed data uploaded at 10 Mbps within a 20 Mbps uplink capacity\n\nRequirements R1–R9 are given along with exact attribute declarations.\n\nThe checker reads only these exact attribute declarations, each placed inside the part usage it belongs to:\n- The vessel part: powerLimitW, uplinkMbps, uploadMbps, storageGB, dataRateMbps, retentionHours\n- Each powered device part: drawW\n\nSo the attributes on the vessel part usage are:\n-

In [16]:
wa.act()


━━ Design A · writer turn 2 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignA {\n    /* Synthetic maritime ISR Design A. Adapted from the MaritimeISR reference model.\n       Numbers are not RTX specif…", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T072814472227/DesignA.sysml

package: DesignA
- 95 line(s), 3006 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T072814472227/DesignA.sysml\n\npackage: DesignA\n- 95 line(s), 3006 characters"}


← ok in 4.6 s
official Pilot PASS · package DesignA · run 20261021T072814-a354282c
- line 39: Duplicate of inherited member name 'lidar' from FixedWingUav
- line 42: Duplicate of inherited member name 'eoCamera' from FixedWingUav
- line 45: Duplicate of inherited member name 'searchRadar' from FixedWingUav
- line 48: Duplicate of inherited member name 'edgeComputer' from FixedWingUav
- line 51: Duplicate of inherited member name 'dataRadio' from FixedWingUav

Returned to the model as the tool's result: 448 characters. The model's next turn decides whether to hand in.


"official Pilot PASS · package DesignA · run 20261021T072814-a354282c\n- line 39: Duplicate of inherited member name 'lidar' from FixedWingUav\n- line 42: Duplicate of inherited member name 'eoCamera' from FixedWingUav\n- line 45: Duplicate of inherited member name 'searchRadar' from FixedWingUav\n- line 48: Duplicate of inherited member name 'edgeComputer' from FixedWingUav\n- line 51: Duplicate of inherited member name 'dataRadio' from FixedWingUav"

In [17]:
wa.decide()


━━ Design A · writer turn 3 · the model reads 4 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 Pilot

 flagged

 the

...

 original

 structure

).


← 8.2 s · read 3196 tokens · wrote 921 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 2828 characters · sha256 82d5f4360e3b


{'content': '',
 'thinking': "The Pilot flagged the redeclarations as duplicates. I'll restructure it so that the device part usages exist only once, inside the top-level `uav` part usage (the part definition will keep only the ports, following the original structure).\n",
 'tool_calls': [{'id': 'gFLWKvGgmyEHD15DixkzmSKHwOGc8gYB',
   'function': {'index': 0,
    'name': 'Validate-SysML-with-the-official-Pilot_tool',
    'arguments': {'flow_tweak_data': {'ChatInput-sysml~input_value': 'package DesignA {\n    /* Synthetic maritime ISR Design A. Adapted from the MaritimeISR reference model.\n       Numbers are not RTX specifications. */\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n    port def TrackPort;\n    port def LinkPort;\n    port def PointCloudPort;\n\n    part def Camera {\n        port imagery : ImageryPort;\n    }\n    part def Lidar {\n        port pointCloud : PointCloudPort;\n    }\n    part def Radar {\n        port tracks : TrackPort;\n    }\n    part

In [18]:
wa.act()


━━ Design A · writer turn 4 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignA {\n    /* Synthetic maritime ISR Design A. Adapted from the MaritimeISR reference model.\n       Numbers are not RTX specif…", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T072827285410/DesignA.sysml

package: DesignA
- 89 line(s), 2827 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T072827285410/DesignA.sysml\n\npackage: DesignA\n- 89 line(s), 2827 characters"}


← ok in 4.6 s
official Pilot PASS · package DesignA · run 20261021T072827-725b9296

Returned to the model as the tool's result: 68 characters. The model's next turn decides whether to hand in.


'official Pilot PASS · package DesignA · run 20261021T072827-725b9296'

Any rounds still needed, then the hand-in. `hand_in()` gives the reply to the pipeline exactly as a Langflow writer's reply would arrive; the pipeline parses it again in ④.

In [19]:
wa.run_until_reply()
wa.hand_in()


━━ Design A · writer turn 5 · the model reads 6 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 model

 passed

.

...

 the

 brief

.


← 14.3 s · read 4114 tokens · wrote 1603 tokens · stop: stop

No tool call: this reply is the design the writer hands in. Run hand_in().

━━ Design A · hand in ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/written {"run": "20261021T072601", "key": "A", "text": "The model passes the official Pilot (run 20261021T072827-725b9296, no diagnostics). Design A is adapted from the MaritimeISR reference: i…", "writer": "notebook: Langflow Agent, Pilot only", "model": "qwen3.8-27b (Ollama)", "tool_calls": "[{'n': 1, 'status': 'PASS', 'sha': '97c5185373b9', 'run_id': '20261021T072814-a354282c'}, {'n': 2, 'status': 'PASS', 'sha': '82d5f4360e3b…", "seconds": 151.1, "model_calls": 3}
← ok in 0.0 s

run 20261021T072601

③ Design A: 3700 characters written by notebook: Langflow Agent, Pilot only on qwen3.8-27b (Ollama); 2 Pilot call(s) by the writer; handed to ④

3 model call(s), 2 Pilot call(s) by the writer (2 PASS), 151.1 s. Handed-i

{'ok': True,
 'run': '20261021T072601',
 'summary': 'run 20261021T072601\n\n③ Design A: 3700 characters written by notebook: Langflow Agent, Pilot only on qwen3.8-27b (Ollama); 2 Pilot call(s) by the writer; handed to ④',
 'data': {'key': 'A', 'characters': 3700, 'written': ['A']},
 '_seconds': 0.0}

### 3.6 ③ Design B · the same, another arrangement

Design B is written new (no similar project to start from), so the writer has only the retrieved cards as examples. If it reaches the card's limit of 20 model calls, `hand_in()` hands in its last `DesignB` draft and says so; ④ then parses it and repairs on its errors with the pipeline's own model, the same second safety net OpenCode's hand-in gets.

In [20]:
wb = pipe.writer("B")


━━ Design pipeline · ③ design B · writer: Langflow Agent, Pilot only chat · Langflow Agent · qwen3.8-flash ━━━━
→ POST http://127.0.0.1:8781/run/prompt {"run": "20261021T072601", "key": "B", "adapt": true}
← ok in 0.0 s

run 20261021T072601

③ prompt for design B: written new · 2869 characters · 5 prompt parts

Card file        langflow/flows/2-coding-agents/langflow-agent-pilot-chat.json (on the canvas this card runs qwen3.8-flash; the pipeline sets the writer's model at run time, here qwen3.8-27b-q3:64k)
Model calls      at most 20 for this design (the card's limit)
Its one tool     Validate-SysML-with-the-official-Pilot_tool → flow “Validate SysML with the official Pilot” (langflow/flows/5-shared-tools/tool-validate-sysml.json): save the text as a file → official Pilot
Branch           written new


In [21]:
wb.decide()


━━ Design B · writer turn 1 · the model reads 2 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › Let

 me

 try

 to

...

 the

 validator

.


← 102.6 s · read 1352 tokens · wrote 10008 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 2542 characters · sha256 d098180641b0


{'content': '',
 'thinking': 'Let me try to carefully parse this task.\n\nI need to write a SysML v2 package named `DesignB` that satisfies the given requirements. Structure:\n- One top-level part usage for the vessel (the UAV)\n- Inside it, one part usage per device\n- Ports and connect statements between devices\n\nThe checker reads only these exact attribute declarations, each placed inside the part usage it belongs to:\n- On the vessel part: `powerLimitW`, `uplinkMbps`, `uploadMbps`, `storageGB`, `dataRateMbps`, `retentionHours`\n- On each powered device part: `drawW`\n\nExact attributes (must use these names verbatim):\n- `attribute powerLimitW : Real = 120;` // R1 — vessel\n- `attribute drawW : Real = 6.5;` // R2 — lidar\n- `attribute drawW : Real = 5;` // R3 — EO camera\n- `attribute drawW : Real = 30;` // R4 — search radar\n- `attribute drawW : Real = 14;` // R5 — edge computer\n- `attribute drawW : Real = 10;` // R6 — data radio\n- `attribute dataRateMbps : Real = 25.6;` // R7

In [22]:
wb.act()


━━ Design B · writer turn 2 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "/**\n * Design B: Fixed-wing MaritimeISR UAV.\n * Variant arrangement: lidar point cloud and EO imagery are routed\n * through a separate lo…", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073028817078/DesignB.sysml

package: DesignB
- 96 line(s), 2541 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073028817078/DesignB.sysml\n\npackage: DesignB\n- 96 line(s), 2541 characters"}


← ok in 3.2 s
official Pilot FAIL · package DesignB · run 20261021T073028-5fa506af
- line 9: no viable alternative at input 'FixedWingUav'
- line 11: no viable alternative at input 'Mid360SLidar'
- line 13: no viable alternative at input 'EoCamera'
- line 15: no viable alternative at input 'SearchRadar'
- line 17: no viable alternative at input 'LocalRecorder'
- line 19: no viable alternative at input 'EdgeComputer'
- line 21: no viable alternative at input 'DataRadio'
- line 24: Couldn't resolve reference to Type 'Real'.
- line 25: Couldn't resolve reference to Type 'Real'.
- line 26: Couldn't resolve reference to Type 'Real'.
- line 27: Couldn't resolve reference to Type 'Real'.
- line 28: Couldn't resolve reference to Type 'Real'.

Returned to the model as the tool's result: 729 characters. The model's next turn decides whether to fix and check again, or hand in.


"official Pilot FAIL · package DesignB · run 20261021T073028-5fa506af\n- line 9: no viable alternative at input 'FixedWingUav'\n- line 11: no viable alternative at input 'Mid360SLidar'\n- line 13: no viable alternative at input 'EoCamera'\n- line 15: no viable alternative at input 'SearchRadar'\n- line 17: no viable alternative at input 'LocalRecorder'\n- line 19: no viable alternative at input 'EdgeComputer'\n- line 21: no viable alternative at input 'DataRadio'\n- line 24: Couldn't resolve reference to Type 'Real'.\n- line 25: Couldn't resolve reference to Type 'Real'.\n- line 26: Couldn't resolve reference to Type 'Real'.\n- line 27: Couldn't resolve reference to Type 'Real'.\n- line 28: Couldn't resolve reference to Type 'Real'."

In [23]:
wb.run_until_reply()
wb.hand_in()


━━ Design B · writer turn 3 · the model reads 4 messages and its one tool, then writes ━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 pilot

 is

 reporting

...

.


← 2.6 s · read 7270 tokens · wrote 207 tokens · stop: stop

The model asks for 1 Pilot call(s) (nothing has run yet; act() runs the first):
  Validate-SysML-with-the-official-Pilot_tool on 44 characters · sha256 0305a9529f29

━━ Design B · writer turn 40 · tool Validate-SysML-with-the-official-Pilot_tool ━━━━━━━━━━━━━━━━━━━━
The flow “Validate SysML with the official Pilot” runs its 2 cards:

[1/2] Save the pasted SysML as a file
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package Test { part P { attribute x = 1; } }", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073328975126/Test.sysml

package: Test
- 1 line(s), 44 characters

[2/2] Official SysML v2 Pilot on that file
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073328975126/Test.sysml\n\npackage: Test\n- 1 line(s), 44 characters"}


← ok in 2.1 s
official Pilot PASS · package Test · run 20261021T073328-9cd53664

Returned to the model as the tool's result: 65 characters. The model's next turn decides whether to hand in.


Stopped: 20 model calls, the card's limit (Langflow's ModelCallLimitMiddleware would stop the card here too).
hand_in() hands in the model's last draft of DesignB (2604 characters); ④ parses it and repairs on its errors.

━━ Design B · hand in ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/written {"run": "20261021T072601", "key": "B", "text": "```sysml\n/**\n * Design B: Fixed-wing MaritimeISR UAV.\n * Variant arrangement: lidar point cloud and EO imagery are routed\n * through a se…", "writer": "notebook: Langflow Agent, Pilot only (stopped at the card's 20-call limit; its last draft is handed in)", "model": "qwen3.8-27b (Ollama)", "tool_calls": "[{'n': 1, 'status': 'FAIL', 'sha': 'd098180641b0', 'run_id': '20261021T073028-5fa506af'}, {'n': 2, 'status': 'FAIL', 'sha': '8e6cd207a82c…", "seconds": 284.9, "model_calls": 20}
← ok in 0.0 s

run 20261021T072601

③ Design B: 2616 characters written by notebook: Langflow Agent,

{'ok': True,
 'run': '20261021T072601',
 'summary': "run 20261021T072601\n\n③ Design B: 2616 characters written by notebook: Langflow Agent, Pilot only (stopped at the card's 20-call limit; its last draft is handed in) on qwen3.8-27b (Ollama); 20 Pilot call(s) by the writer; handed to ④",
 'data': {'key': 'B', 'characters': 2616, 'written': ['A', 'B']},
 '_seconds': 0.0}

### 3.7 What the writer was given

The three read-only cards on the canvas: the rules, the attribute template, and the examples or the similar project.

In [24]:
pipe.prompt_cards()


━━ Design pipeline · ③ shown · Rules · Paper 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/tool/prompt-part {"run": "20261021T072601", "part": "rules"}
← ok in 0.0 s
run 20261021T072601 · Rules (design A, paper p3)

You write SysML v2 textual notation.

Rules:

- Output one fenced code block tagged sysml, and no prose before or after it.

- The file contains exactly one package whose name is given in the task.

- When you use Real, write `private import ScalarValues::*;` as the first line inside the package. In this toolchain an import inside a package needs the visibility keyword; a bare `import` is rejected. Do not import anything else.

- Use only names the task requires. Do not invent numeric limits.

- Do not change a number the task tells you to keep.

- The first line inside the package is exactly: private import ScalarValues::*;

- Declare attributes as `attribute name : Real = 12.0;` inside part usages, not part definitions, so values ar

### 3.8 Every Pilot call the writer made, A and B

In order, with the package each call was on: the writer may test small packages of its own between drafts, and those show up here as calls on other packages.

In [25]:
pipe.pilot_rounds()


━━ Design A · every Pilot call by the writer (2: 2 on DesignA, 0 on test packages), before the pipeline's own check ━━━━
Pilot call #1: package DesignA · PASS · sha256 97c5185373b9 · Pilot run 20261021T072814-a354282c
Pilot call #2: package DesignA · PASS · sha256 82d5f4360e3b · Pilot run 20261021T072827-725b9296
handed in: 3700 characters · sha256 of its sysml block 82d5f4360e3b

━━ Design B · every Pilot call by the writer (20: 3 on DesignB, 17 on test packages), before the pipeline's own check ━━━━
Pilot call #1: package DesignB · FAIL · sha256 d098180641b0 · Pilot run 20261021T073028-5fa506af
      line 9: no viable alternative at input 'FixedWingUav'
      line 11: no viable alternative at input 'Mid360SLidar'
      line 13: no viable alternative at input 'EoCamera'
      line 15: no viable alternative at input 'SearchRadar'
      line 17: no viable alternative at input 'LocalRecorder'
      line 19: no viable alternative at input 'EdgeComputer'
Pilot call #2: package DesignB · F

### 3.9 ④ The pipeline checks again with the official Pilot

The pipeline does not take the writer's own Pilot calls as the verdict. It takes the ```sysml block of each reply and runs the official Pilot on it again; on a failure the complete error list goes back to the pipeline's model, up to 20 repairs. The cell also says whether the text it parsed is the text the writer's last Pilot call passed.

In [26]:
pipe.validate()


━━ Design pipeline · ④ Official SysML v2 Pilot ⇄ repair (the pipeline's own check) ━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/validate {"run": "20261021T072601"}


← ok in 20.4 s

run 20261021T072601

④ Official SysML v2 Pilot, with repair from errors (≤ 20)
- Design A: PASS after 1 parse(s), 0 repair(s)
- attempt 1: PASS
- Design B: PASS after 2 parse(s), 1 repair(s)
- attempt 1: FAIL · Couldn't resolve reference to Type 'std::Real'.
- attempt 2: PASS

Design A: the pipeline took the ```sysml block of the writer's reply and ran the official Pilot on it.
  attempt 1: PASS · Pilot run 20261021T073331-b60a5e65
  repairs after hand-in: 0
  The writer's last Pilot call on DesignA (2 on this design, 2 PASS) said PASS on text sha256 82d5f4360e3b. The text the pipeline parsed (sha256 82d5f4360e3b) is that same text.

Design B: the pipeline took the ```sysml block of the writer's reply and ran the official Pilot on it.
  attempt 1: FAIL · Pilot run 20261021T073335-a768c1dd
      line 24: Couldn't resolve reference to Type 'std::Real'.
      line 25: Couldn't resolve reference to Type 'std::Real'.
      line 26: Couldn't resolve reference to Type 'std::Re

{'ok': True,
 'run': '20261021T072601',
 'summary': "run 20261021T072601\n\n④ Official SysML v2 Pilot, with repair from errors (≤ 20)\n- Design A: PASS after 1 parse(s), 0 repair(s)\n- attempt 1: PASS\n- Design B: PASS after 2 parse(s), 1 repair(s)\n- attempt 1: FAIL · Couldn't resolve reference to Type 'std::Real'.\n- attempt 2: PASS",
 'data': [{'key': 'A',
   'sysml_status': 'PASS',
   'attempts': [{'attempt': 1,
     'status': 'PASS',
     'diagnostics': [],
     'sha': '82d5f4360e3b',
     'same_as_previous': False,
     'run_id': '20261021T073331-b60a5e65',
     'text': 'package DesignA {\n    /* Synthetic maritime ISR Design A. Adapted from the MaritimeISR reference model.\n       Numbers are not RTX specifications. */\n    private import ScalarValues::*;\n\n    port def ImageryPort;\n    port def TrackPort;\n    port def LinkPort;\n    port def PointCloudPort;\n\n    part def Camera {\n        port imagery : ImageryPort;\n    }\n    part def Lidar {\n        port pointCloud : P

### 3.10 ⑤ Align with memory · Paper 1

In [27]:
pipe.align()


━━ Design pipeline · ⑤ Align with memory · Paper 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/align {"run": "20261021T072601"}


← ok in 10.7 s

run 20261021T072601

⑤ Alignment with MaritimeISR (paper 1): 4 pairs proposed, 4 with both names present, 4 declared; package AlignA_20261021T072601 official Pilot PASS
- DesignA::FixedWingUav → MaritimeISR::FixedWingUav · alias
- DesignA::uav → MaritimeISR::uav · allocation
- DesignA::DataRadio → MaritimeISR::FixedWingUav::radio · alias
- DesignA::uav::uplinkMbps → MaritimeISR::FixedWingUav::downlinkMbps · alias
- Mappings are proposals. Code checks both names exist, writes alias and allocation statements from the two parses, then the official Pilot parses the alignment package. Neither source model is edited. A person accepts or rejects each pair.


{'ok': True,
 'run': '20261021T072601',
 'summary': 'run 20261021T072601\n\n⑤ Alignment with MaritimeISR (paper 1): 4 pairs proposed, 4 with both names present, 4 declared; package AlignA_20261021T072601 official Pilot PASS\n- DesignA::FixedWingUav → MaritimeISR::FixedWingUav · alias\n- DesignA::uav → MaritimeISR::uav · allocation\n- DesignA::DataRadio → MaritimeISR::FixedWingUav::radio · alias\n- DesignA::uav::uplinkMbps → MaritimeISR::FixedWingUav::downlinkMbps · alias\n- Mappings are proposals. Code checks both names exist, writes alias and allocation statements from the two parses, then the official Pilot parses the alignment package. Neither source model is edited. A person accepts or rejects each pair.',
 'data': {'mappings': [{'design': 'DesignA::FixedWingUav',
    'existing': 'MaritimeISR::FixedWingUav',
    'reason': 'Both represent the fixed-wing unmanned aerial vehicle platform.',
    'design_exists': True,
    'existing_exists': True,
    'ok': True,
    'construct': 'alias

### 3.11 ⑥ Checks and read-back

In [28]:
pipe.checks()


━━ Design pipeline · ⑥ Checks + read-back ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/checks {"run": "20261021T072601"}


← ok in 3.4 s

fail · run 20261021T072601

⑥ Checks on numbers read from the official parse; read-back (paper 3 query)
- Design A: SOURCE PASS (11 / 11 numbers), POWER PASS (65.5 / 120.0 W), LINK PASS (10.0 / 20.0 Mbps), STORE PASS (92.16 / 256.0 GB), TRACE PASS (9 / 9 reqs), REQS UNKNOWN, MISSION PASS
- read-back: missing 0, 9, 20, 27, 100, 120, 200
- Design B: SOURCE PASS (11 / 11 numbers), POWER PASS (65.5 / 120.0 W), LINK PASS (10.0 / 20.0 Mbps), STORE PASS (92.16 / 256.0 GB), TRACE PASS (9 / 9 reqs), REQS UNKNOWN, MISSION PASS
- read-back: missing 0, 9, 20, 27, 100, 120, 200
- eligible for approval: none


{'ok': True,
 'run': '20261021T072601',
 'summary': 'fail · run 20261021T072601\n\n⑥ Checks on numbers read from the official parse; read-back (paper 3 query)\n- Design A: SOURCE PASS (11 / 11 numbers), POWER PASS (65.5 / 120.0 W), LINK PASS (10.0 / 20.0 Mbps), STORE PASS (92.16 / 256.0 GB), TRACE PASS (9 / 9 reqs), REQS UNKNOWN, MISSION PASS\n- read-back: missing 0, 9, 20, 27, 100, 120, 200\n- Design B: SOURCE PASS (11 / 11 numbers), POWER PASS (65.5 / 120.0 W), LINK PASS (10.0 / 20.0 Mbps), STORE PASS (92.16 / 256.0 GB), TRACE PASS (9 / 9 reqs), REQS UNKNOWN, MISSION PASS\n- read-back: missing 0, 9, 20, 27, 100, 120, 200\n- eligible for approval: none',
 'verdict': 'fail',
 'data': {'failing': [],
  'eligible': [],
  'designs': [{'key': 'A',
    'checks': [{'id': 'SOURCE',
      'label': 'Every design number is in the brief',
      'formula': 'each parsed literal appears in the brief or in a sourced requirement',
      'status': 'PASS',
      'actual': 11,
      'limit': 11,
      'u

### 3.12 Router: all checks pass?

In [29]:
pipe.route_checks()


━━ Design pipeline · Router “All four checks pass?” ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The router reads the checks card's text and asks: does it start with “pass”?
It starts with “fail” → False: Trade-offs, then ⑦ Review.


False

### 3.13 Trade-offs (only when a check failed)

In [30]:
pipe.tradeoffs()


━━ Design pipeline · Trade-offs (catalog parts only; code recomputes each) ━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/tradeoffs {"run": "20261021T072601"}
← ok in 0.0 s

run 20261021T072601

Trade-offs for: nothing failed (catalog parts only; code recomputes each)


{'ok': True,
 'run': '20261021T072601',
 'summary': 'run 20261021T072601\n\nTrade-offs for: nothing failed (catalog parts only; code recomputes each)',
 'data': {'proposals': []},
 '_seconds': 0.0}

### 3.14 ⑦ Review

The flow's last card. Its text is what the Agent receives as the tool's result.

In [31]:
pipe.review()


━━ Design pipeline · ⑦ Diagram · Cameo export · review text ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:8781/run/review {"run": "20261021T072601"}


← ok in 0.2 s

run 20261021T072601 · review for a person

Brief: Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.

Design A: official Pilot PASS after 1 parse(s); SOURCE PASS, POWER PASS, LINK PASS, STORE PASS, TRACE PASS, REQS UNKNOWN, MISSION PASS
- approval: The brief leaves a value open and the requirements ask for it; it cannot go into memory.
- ⑦ diagram: http://127.0.0.1:8781/diagram?run=20261021T073331-b60a5e65&view=connections
- Cameo: export ready at analysis/cameo-exports/20261021T072601/DesignA.sysml (open it in Cameo's SysML v2 when RTRC


This text is the flow's output (ChatOutput-review): Langflow hands it to the Agent as Design-pipeline_tool's result.


"run 20261021T072601 · review for a person\n\nBrief: Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.\n\nDesign A: official Pilot PASS after 1 parse(s); SOURCE PASS, POWER PASS, LINK PASS, STORE PASS, TRACE PASS, REQS UNKNOWN, MISSION PASS\n- approval: The brief leaves a value open and the requirements ask for it; it cannot go into memory.\n- ⑦ diagram: http://127.0.0.1:8781/diagram?run=20261021T073331-b60a5e65&view=connections\n- Cameo: export ready at analysis/cameo-exports/20261021T072601/DesignA.sysml (open it in Cameo's SysML v2 when RTRC provid

## 4 · The agent reads the review and answers

In [32]:
agent.give_back(pipe)


━━ Step 9 · the Design pipeline flow hands its last card's text back to the Agent ━━━━━━━━━━━━━━━━━━
1,663 characters (the review shown in ⑦) go back to the model as the tool's result.


In [33]:
agent.run_until_reply()


━━ Step 10 · the model reads 11 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 execution

 has

 completed

...

 want

 to

 proceed

?


← 12.0 s · read 6510 tokens · wrote 795 tokens · stop: stop

No tool call: the text after “answer ›” is the agent's reply, as the Playground shows it.


## 5 · Turn 3 · the approval card

In Langflow the Agent stops before Write to memory and shows Approve and Reject. Here `APPROVE` decides. It is `False`, so nothing is written; set it to `True` to store the design.

In [34]:
agent.say("Approve design B.")


━━ Engineer ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Approve design B.


In [35]:
agent.run_until_reply()


━━ Step 11 · the model reads 13 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 engineer

 is

 trying

...

0

1

 B

".


← 12.5 s · read 7033 tokens · wrote 1115 tokens · stop: stop

The model chose (nothing has run yet; agent.act() runs it):
  Write-to-memory_tool({"flow_tweak_data": {"ChatInput-approval~input_value": "20261021T072601 B"}})

━━ Step 12 · tool Write-to-memory_tool · canvas card “Tool · write to memory (a flow) · approval card · Paper 2” ━━━━
arguments: {"flow_tweak_data": {"ChatInput-approval~input_value": "20261021T072601 B"}}

Langflow stops here and shows “Tool execution requires approval” with Approve and Reject.
Nothing runs until agent.approve(True) (run it) or agent.approve(False) (refuse; the model is told).


In [36]:
APPROVE = False
agent.approve(APPROVE)


━━ Step 13 · the person refused Write-to-memory_tool ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Nothing ran. The model is told the person rejected the call.

Returned to the model as Write-to-memory_tool's result: 51 characters.


In [37]:
agent.run_until_reply()


━━ Step 14 · the model reads 15 messages and the 9 tools, then chooses ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
→ POST http://127.0.0.1:11434/api/chat · model qwen3.8-27b-q3:64k · thinking switch: not sent (Ollama default)



thinking › The

 write

-to

-memory

...

'll

 record

 it

.


← 4.5 s · read 7120 tokens · wrote 354 tokens · stop: stop

No tool call: the text after “answer ›” is the agent's reply, as the Playground shows it.


## 6 · Check it yourself: the official Pilot on this run's files

In [38]:
pipe.recheck()


━━ Check it yourself · official Pilot on mvp/exam-inbox/discover-20261021T072601/DesignA.sysml ━━━━━
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignA {\n    /* Synthetic maritime ISR Design A. Adapted from the MaritimeISR reference model.\n       Numbers are not RTX specif…", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073434995921/DesignA.sysml

package: DesignA
- 89 line(s), 2827 characters
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073434995921/DesignA.sysml\n\npackage: DesignA\n- 89 line(s), 2827 characters"}


← ok in 4.2 s
official Pilot PASS · package DesignA · run 20261021T073434-53ed2e8e

━━ Check it yourself · official Pilot on mvp/exam-inbox/discover-20261021T072601/DesignB.sysml ━━━━━
→ POST http://127.0.0.1:8781/tool/validate/save {"sysml": "package DesignB {\n  private import ScalarValues::*;\n\n  part def FixedWingUav {}\n\n  part def Mid360SLidar {}\n\n  part def EoCamera {}\n\n  pa…", "upstream": ""}
← ok in 0.0 s
file: mvp/exam-inbox/assistant-20261021T073439236305/DesignB.sysml

package: DesignB
- 91 line(s), 2343 characters
→ POST http://127.0.0.1:8781/tool/validate/pilot {"saved": "file: mvp/exam-inbox/assistant-20261021T073439236305/DesignB.sysml\n\npackage: DesignB\n- 91 line(s), 2343 characters"}


← ok in 4.4 s
official Pilot PASS · package DesignB · run 20261021T073439-ebde7fc3


## 7 · The conversation as the Playground lists it, and where the files are

In [39]:
agent.transcript()
pipe.files()


━━ The conversation, as the Playground lists it ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Engineer › I want a new design for this fixed-wing UAV. First check the memory for anything similar, then show me the exact brief you would send to the design pipeline.

Design a fixed-wing UAV modeled on the MaritimeISR FixedWingUav structure, carrying a Mid-360S lidar (6.5 W, 9–27 V DC, 100 BASE-TX Ethernet, 200,000 points/s at 10 Hz), an EO camera (5 W) and a search radar (30 W), all feeding an edge computer (14 W) and a data radio (10 W), under a total power limit of 120 W. The lidar point cloud shall be recorded locally at 25.6 Mbps with 8 h retention on 256 GB of storage. Processed data shall be uploaded at 10 Mbps within a 20 Mbps uplink capacity.
  Agent called Memory-search_tool({"flow_tweak_data": {"ChatInput-query~input_value": "fixed-wing UAV lidar EO camera search radar edge computer data radio power storage upload"}})
  Agent called list_projects({})
    ↳ Memory-search_

## Appendix · the canvas's writer, OpenCode

To see the writer the Design pipeline card actually uses, replace the cells of 3.5 and 3.6 with `pipe.design()`: OpenCode writes A and B in the sandbox and the cell prints each file it writes and each `./validate` it runs as they happen; `pipe.opencode("A")` then replays its whole session from its event log. Do not run both writers on one run. Run `20261021T063412` on results page 2.3 was made that way.